In [1]:
import pandas as pd
import os

# AOD filename key → meteo filename key
sites = {
    "GSFC":   "NASA_Goddard",
    "HU_IRB": "HU_IRB",
    "SERC":   "SERC",
}

out_dir = "../../data/merged_aod_metero_1"
os.makedirs(out_dir, exist_ok=True)

for aod_key, meteo_key in sites.items():
    # --- Load AOD (UTC) ---
    aod = pd.read_csv(f"../../data/{aod_key}_AOD500_preprocessed.csv",
                      parse_dates=["datetime"])
    aod["datetime"] = aod["datetime"].dt.tz_localize("UTC").dt.tz_convert("America/New_York")
    aod["datetime"] = aod["datetime"].dt.tz_localize(None)   # drop tz info for merge
    aod = aod.sort_values("datetime")
    print(f"length initially: {len(aod)}")

    # --- Load meteo (already local EST, no tz info) ---
    meteo = pd.read_csv(f"../../data/metero/{meteo_key}_hourly_meteo.csv",
                        parse_dates=["DateTime"])
    meteo = meteo.sort_values("DateTime")

    # --- Merge: match each AOD obs to nearest hourly meteo row ---
    merged = pd.merge_asof(
        aod,
        meteo,
        left_on="datetime",
        right_on="DateTime",
        direction="nearest",
        tolerance=pd.Timedelta("30min")   # only match if within ±30 min
    )

    out_path = os.path.join(out_dir, f"{aod_key}_merged.csv")
    merged.to_csv(out_path, index=False)
    print(f"{aod_key}: {len(aod)} AOD rows → {len(merged)} merged rows saved to {out_path}")
    print(f"length after merging: {len(merged)}")


length initially: 66337
GSFC: 66337 AOD rows → 66337 merged rows saved to ../../data/merged_aod_metero_1/GSFC_merged.csv
length after merging: 66337
length initially: 27972
HU_IRB: 27972 AOD rows → 27972 merged rows saved to ../../data/merged_aod_metero_1/HU_IRB_merged.csv
length after merging: 27972
length initially: 39393
SERC: 39393 AOD rows → 39393 merged rows saved to ../../data/merged_aod_metero_1/SERC_merged.csv
length after merging: 39393


In [3]:
from IPython.display import display
df = pd.read_csv("../../data/merged_aod_metero_1/HU_IRB_merged.csv")
display(df.head())
print(df.shape)

df = pd.read_csv("../../data/HU_IRB_AOD500_preprocessed.csv")
display(df.head())
print(df.shape)


,datetime,AOD_500nm,DateTime,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m,Site
0,2021-03-01 13:56:43,0.163369,2021-03-01 14:00:00,8.0,75,0.0,22.7,306,HU IRB
1,2021-03-01 14:01:43,0.144799,2021-03-01 14:00:00,8.0,75,0.0,22.7,306,HU IRB
2,2021-03-01 15:16:42,0.170261,2021-03-01 15:00:00,9.1,60,0.0,23.1,301,HU IRB
3,2021-03-01 15:21:42,0.179449,2021-03-01 15:00:00,9.1,60,0.0,23.1,301,HU IRB
4,2021-03-01 15:30:52,0.152524,2021-03-01 16:00:00,9.4,53,0.0,22.4,300,HU IRB


(27972, 9)


,datetime,AOD_500nm
0,2021-03-01 18:56:43,0.163369
1,2021-03-01 19:01:43,0.144799
2,2021-03-01 20:16:42,0.170261
3,2021-03-01 20:21:42,0.179449
4,2021-03-01 20:30:52,0.152524


(27972, 2)
